# Enhanced Allergen Labeling & Annotation

**Purpose:** This notebook processes and analyzes food label data for allergen detection.


## Overview
Annotates ingredients with allergen labels for model training.

## Workflow
1. Define allergen categories
2. Create mapping rules
3. Label ingredients
4. Generate training dataset


In [1]:
import pandas as pd
import ast
import re
from collections import defaultdict

# Load the cleaned dataset
df = pd.read_csv("../data/processed/cleaned_dataset.csv")

In [2]:
def clean_ingredient_text(text):
# Remove allergen declarations and cross-contamination statements.
    if not isinstance(text, str):
        return text
    # Patterns to remove (case-insensitive)
    patterns = [
        r'allergen information[^.]*\.',
        r'may contain[^.]*\.',
        r'contains\s+(?:wheat|milk|eggs|soy|fish|shellfish|peanuts|tree nuts)[^.]*\.',
        r'free from[^.]*\.',
        r'does not contain[^.]*\.',
        r'contains no[^.]*\.'
    ]
    for pat in patterns:
        text = re.sub(pat, '', text, flags=re.IGNORECASE)
    return text.strip()

# Apply cleaning BEFORE tokenizing
df["ingredients_text_cleaned"] = df["ingredients_text_en"].apply(clean_ingredient_text)

# Re-create tokens from cleaned text
df["ingredient_tokens_cleaned"] = df["ingredients_text_cleaned"].apply(
    lambda x: re.findall(r'\b[\w\-]+\b', x.lower()) if isinstance(x, str) else []
)

# Use cleaned tokens for detection (keep original if needed)
df["ingredient_tokens_original"] = df["ingredient_tokens"]
df["ingredient_tokens"] = df["ingredient_tokens_cleaned"]

In [3]:
# Clean ingredient tokens: remove non‑ingredient noise (allergen statements, storage instructions, etc.)
import re

def clean_tokens(tokens):
    # Join tokens into a single string
    text = " ".join(tokens).lower()
    # Remove common non‑ingredient phrases
    noise_patterns = [
        r'allergen information[^.]*\.',
        r'may contain[^.]*\.',
        r'contains[:\s]+[^.]+\.',
        r'store in[^.]*\.',
        r'keep away[^.]*\.',
        r'manufactured[^.]*\.',
        r'produced in[^.]*\.',
        r'for[^.]*allergy[^.]*\.',
        r'product of[^.]*\.',
        r'best before[^.]*\.',
        r'exp[^.]*\.',
    ]
    for pat in noise_patterns:
        text = re.sub(pat, '', text, flags=re.IGNORECASE)
    # Re‑tokenize on whitespace and punctuation
    cleaned = re.findall(r'\b\w+(?:[ -]\w+)*\b', text)
    return cleaned

# Apply cleaning
df["ingredient_tokens_cleaned"] = df["ingredient_tokens"].apply(clean_tokens)

# Use cleaned tokens for detection (you can either replace or keep original)
# We'll keep both and modify the detection function to accept a parameter

In [4]:
# Expanded allergen keywords (coconut removed from tree_nuts, handled separately)
BIG8 = {
    "milk": [
        "milk", "whey", "casein", "caseinate", "butter", "cheese",
        "lactose", "cream", "ghee", "buttermilk", "milk solids", "skim milk",
        "condensed milk", "powdered milk", "nonfat milk", "curds", "yogurt",
        "kefir", "lactalbumin", "lactoglobulin"
    ],
    "eggs": [
        "egg", "albumin", "egg yolk", "egg white", "lysozyme", "ovalbumin",
        "mayonnaise", "meringue", "ovomucoid", "ovotransferrin", "eggshell"
    ],
    "peanuts": [
        "peanut", "groundnut", "peanut oil", "peanut butter", "arachis oil",
        "monkey nut", "goober", "beer nut", "peanut flour", "peanut protein"
    ],
    "tree_nuts": [
        "almond", "cashew", "walnut", "pecan", "hazelnut", "filbert",
        "pistachio", "macadamia", "brazil nut", "pine nut", "chestnut",
        "nut paste", "nut butter", "nut meal", "nut oil", "shea nut"
    ],
    "soy": [
        "soy", "soya", "soybean", "soy lecithin", "tofu", "miso", "edamame",
        "textured vegetable protein", "tvp", "soy protein", "soy sauce",
        "tamari", "tempeh", "soy milk", "soybean oil", "soy flour"
    ],
    "wheat": [
        "wheat", "gluten", "semolina", "durum", "farina", "breadcrumbs",
        "spelt", "triticale", "bran", "cereal", "flour", "wheat germ",
        "wheat starch", "bulgur", "couscous", "einkorn", "kamut"
    ],
    "fish": [
        "fish", "salmon", "tuna", "anchovy", "cod", "pollock", "fish sauce",
        "surimi", "sardine", "mackerel", "trout", "roe", "fish oil",
        "fish gelatin", "isnglass", "caviar", "fish meal", "omega-3 from fish"
    ],
    "shellfish": [
        "shrimp", "crab", "lobster", "prawn", "krill", "crayfish",
        "squid", "mussel", "clam", "oyster", "scallop", "crab paste",
        "langoustine", "crawfish", "abalone", "conch", "cockle", "whelk",
        "shellfish extract", "crab meat", "lobster paste"
    ],
    "coconut": [
        "coconut", "coconut oil", "coconut milk", "coconut flour"
    ],
}

# Mapping from official tag suffix to Big-8 key
OFFICIAL_MAP = {
    "gluten": "wheat",
    "wheat": "wheat",
    "milk": "milk",
    "eggs": "eggs",
    "soybeans": "soy",
    "soya": "soy",
    "peanuts": "peanuts",
    "tree nuts": "tree_nuts",
    "nuts": "tree_nuts",
    "fish": "fish",
    "shellfish": "shellfish"
}

In [5]:
def detect_allergens(tokens, use_cleaned=True):
    """
    Detect allergens from ingredient tokens.
    If use_cleaned=True, expects cleaned tokens (without noise).
    """
    # Join tokens into a single string
    text = " ".join(tokens).lower()
    # Preprocess: remove punctuation except spaces
    text_clean = re.sub(r'[^\w\s]', ' ', text)
    
    # ---- 1. Negative patterns (free, without, no, etc.) ----
    negative_patterns = [
        r'\b(?:free|without|no|zero|minus|不含|無)\s+(\w+)',
        r'\b(\w+)\s*-free\b',
        r'does not contain\s+(\w+)',
        r'contains no\s+(\w+)',
        r'allergen free\s+(\w+)',
    ]
    allergens_to_avoid = set()
    for pattern in negative_patterns:
        matches = re.findall(pattern, text_clean)
        for match in matches:
            if isinstance(match, tuple):
                match = match[0]
            for allergen, keywords in BIG8.items():
                for kw in keywords:
                    if kw in match or match in kw:
                        allergens_to_avoid.add(allergen)
    
    # ---- 2. Exempt forms (highly refined or safe derivatives) ----
    exempt_patterns = {
        "peanuts": [r'\bpeanut oil\b', r'\barachis oil\b', r'\bpeanut flour\b'],
        "soy": [r'\bsoy lecithin\b', r'\bsoybean oil\b', r'\bsoya oil\b', r'\bhydrolyzed soy protein\b'],
        "fish": [r'\bfish oil\b', r'\bomega-3 from fish\b'],
        "tree_nuts": [r'\bcoconut oil\b', r'\bcoconut milk\b', r'\bshea butter\b'],
        "wheat": [r'\bwheat starch\b', r'\bgluten-free wheat starch\b'],
    }
    
    found = set()
    
    # ---- 3. Detection loop with improved context ----
    for allergen, keywords in BIG8.items():
        if allergen in allergens_to_avoid:
            continue
        
        # Special case: "coconut milk" / "soy milk" should NOT trigger "milk"
        if allergen == "milk" and re.search(r'\b(?:coconut|soy|almond|oat|rice)\s+milk\b', text_clean):
            continue
        
        # Special case for wheat: require explicit wheat terms unless "wheat flour"
        if allergen == "wheat":
            wheat_terms = ['wheat', 'gluten', 'durum', 'semolina', 'spelt', 'triticale', 'bulgur', 'couscous', 'einkorn', 'kamut']
            if any(re.search(r'\b' + re.escape(w) + r'\b', text_clean) for w in wheat_terms):
                # explicit wheat – ok
                pass
            else:
                # Only "flour" or "cereal" – require "wheat flour" or "wheat cereal"
                if re.search(r'\bflour\b', text_clean) and not re.search(r'\bwheat\s+flour\b', text_clean):
                    continue
                if re.search(r'\bcereal\b', text_clean) and not re.search(r'\bwheat\s+cereal\b', text_clean):
                    continue
        
        # Check for any keyword match
        matched = False
        for kw in keywords:
            pattern = r'\b' + re.escape(kw) + r's?\b'
            if re.search(pattern, text_clean):
                matched = True
                break
        
        if not matched:
            continue
        
        # ---- Apply exemption: if only exempt forms are present, skip ----
        if allergen in exempt_patterns:
            exempt_matches = any(re.search(pat, text_clean) for pat in exempt_patterns[allergen])
            # Non‑exempt keywords for this allergen
            non_exempt_keywords = [kw for kw in keywords if not any(re.search(pat, kw) for pat in exempt_patterns.get(allergen, []))]
            non_exempt_matches = any(re.search(r'\b' + re.escape(kw) + r's?\b', text_clean) for kw in non_exempt_keywords)
            if exempt_matches and not non_exempt_matches:
                continue
        
        found.add(allergen)
    
    return list(found)

# Apply detection using cleaned tokens (if you created the cleaning cell)
if "ingredient_tokens_cleaned" in df.columns:
    df["detected_allergens"] = df["ingredient_tokens_cleaned"].apply(detect_allergens)
else:
    df["detected_allergens"] = df["ingredient_tokens"].apply(detect_allergens)

In [6]:
# Separate coconut detection (coconut is not a Big-8 allergen, but useful)
coconut_keywords = ["coconut", "coconut oil", "coconut milk", "coconut flour"]
def detect_coconut(tokens):
    text = " ".join(tokens).lower()
    if any(re.search(r'\b' + re.escape(kw) + r's?\b', text) for kw in coconut_keywords):
        return ["coconut"]
    return []

df["coconut"] = df["ingredient_tokens"].apply(detect_coconut)

In [7]:
def apply_exemptions(detected_list, tokens):
    text = " ".join(tokens).lower()
    detected_set = set(detected_list)
    
    exempt_config = {
        "soy": {
            "patterns": [r'\bsoy lecithin\b', r'\bsoybean oil\b', r'\bsoya oil\b', r'\bhydrolyzed soy protein\b'],
            "keep_if_also": [r'\bsoy protein\b', r'\btofu\b', r'\bmiso\b', r'\btempeh\b', r'\bsoy sauce\b']
        },
        "wheat": {
            "patterns": [r'\bflour\b', r'\bcereal\b', r'\bbran\b', r'\bwheat starch\b'],
            "keep_if_also": [r'\bwheat\b', r'\bgluten\b', r'\bspelt\b', r'\bdurum\b', r'\bsemolina\b']
        },
        "coconut": {
            "patterns": [r'\bcoconut oil\b', r'\bcoconut milk\b', r'\bcoconut flour\b'],
            "keep_if_also": [r'\bcoconut\b(?!\s+(?:oil|milk|flour))']
        },
        "tree_nuts": {
            "patterns": [r'\bshea butter\b', r'\bcoconut oil\b', r'\bcoconut milk\b'],
            "keep_if_also": [r'\b(?:almond|cashew|walnut|pecan|hazelnut|pistachio|macadamia|brazil nut|pine nut|chestnut)\b']
        },
        "eggs": {
            "patterns": [r'\blecithin\b'],  # egg lecithin is rare; if no other egg word, skip
            "keep_if_also": [r'\begg\b', r'\begg yolk\b', r'\begg white\b', r'\balbumin\b']
        }
    }
    
    for allergen, config in exempt_config.items():
        if allergen not in detected_set:
            continue
        exempt_match = any(re.search(pat, text) for pat in config["patterns"])
        if not exempt_match:
            continue
        keep_match = any(re.search(pat, text) for pat in config.get("keep_if_also", []))
        if exempt_match and not keep_match:
            detected_set.discard(allergen)
    
    return list(detected_set)

# Apply exemptions (using cleaned tokens if available)
if "ingredient_tokens_cleaned" in df.columns:
    df["detected_allergens"] = df.apply(
        lambda row: apply_exemptions(row["detected_allergens"], row["ingredient_tokens_cleaned"]), axis=1
    )
else:
    df["detected_allergens"] = df.apply(
        lambda row: apply_exemptions(row["detected_allergens"], row["ingredient_tokens"]), axis=1
    )

print("Exemptions applied (expanded).")

Exemptions applied (expanded).


In [8]:
def parse_traces_tags(tag_str):
    try:
        tags = ast.literal_eval(tag_str)
    except:
        return []
    mapped = []
    for tag in tags:
        if not tag.startswith('en:'):
            continue
        suffix = tag.split(':', 1)[1]
        mapped_key = OFFICIAL_MAP.get(suffix)
        if mapped_key:
            mapped.append(mapped_key)
        elif suffix == "tree-nuts":
            mapped.append("tree_nuts")
    return list(set(mapped))

df["traces_allergens"] = df["traces_tags"].apply(parse_traces_tags)
print("Traces column added.")

Traces column added.


In [9]:
# Parse official allergen tags from the 'allergens_tags' column
def parse_official_tags(tag_str):
    try:
        tags = ast.literal_eval(tag_str)
    except:
        return []
    mapped = []
    for tag in tags:
        if not tag.startswith('en:'):
            continue
        suffix = tag.split(':', 1)[1]
        mapped_key = OFFICIAL_MAP.get(suffix)
        if mapped_key:
            mapped.append(mapped_key)
        elif suffix == "tree-nuts":
            mapped.append("tree_nuts")
    return list(set(mapped))

df["official_allergens_mapped"] = df["allergens_tags"].apply(parse_official_tags)
print("Official allergens mapped.")

Official allergens mapped.


In [10]:
def extract_may_contain(text):
    if not isinstance(text, str):
        return []
    pattern = r'may contain\s*:?\s*([^\.]+)'
    match = re.search(pattern, text.lower())
    if match:
        candidates = match.group(1)
        found = []
        for allergen in BIG8.keys():
            if allergen.replace('_', ' ') in candidates or allergen in candidates:
                found.append(allergen)
        return found
    return []

df["may_contain"] = df["ingredients_text_en"].apply(extract_may_contain)

In [11]:
# Identify products that explicitly list allergens (more reliable ground truth)
def has_explicit_allergen_statement(text):
    if not isinstance(text, str):
        return False
    return bool(re.search(r'contains\s*:\s*\w+', text, re.IGNORECASE) or
                re.search(r'allergen information\s*:\s*\w+', text, re.IGNORECASE))

df["explicit_allergen_statement"] = df["ingredients_text_en"].apply(has_explicit_allergen_statement)
eval_df = df[df["explicit_allergen_statement"]]

if len(eval_df) > 0:
    print(f"Evaluating on {len(eval_df)} products with explicit allergen statements (out of {len(df)} total).\n")
    
    strict = (eval_df["detected_allergens"].apply(set) == eval_df["official_allergens_mapped"].apply(set)).mean()
    subset = (eval_df["detected_allergens"].apply(set) <= eval_df["official_allergens_mapped"].apply(set)).mean()
    superset = (eval_df["detected_allergens"].apply(set) >= eval_df["official_allergens_mapped"].apply(set)).mean()
    
    def jaccard(s1, s2):
        if len(s1) == 0 and len(s2) == 0:
            return 1.0
        inter = len(s1 & s2)
        union = len(s1 | s2)
        return inter / union if union > 0 else 0
    
    avg_jaccard = eval_df.apply(lambda row: jaccard(set(row["detected_allergens"]), set(row["official_allergens_mapped"])), axis=1).mean()
    
    print(f"Exact match agreement (strict, filtered): {strict:.2%}")
    print(f"Detected ⊆ official (no false positives): {subset:.2%}")
    print(f"Detected ⊇ official (no false negatives): {superset:.2%}")
    print(f"Average Jaccard similarity: {avg_jaccard:.2%}")
else:
    print("No products with explicit allergen statements found.")

Evaluating on 96 products with explicit allergen statements (out of 1057 total).

Exact match agreement (strict, filtered): 17.71%
Detected ⊆ official (no false positives): 20.83%
Detected ⊇ official (no false negatives): 96.88%
Average Jaccard similarity: 18.75%


In [12]:
def combine_allergen_labels(detected, official, traces, may_contain):
    
    detected_set = set(detected)
    official_set = set(official)
    traces_set = set(traces)
    may_contain_set = set(may_contain)
    
    return {
        "detected_only": sorted(detected_set),
        "detected_or_official": sorted(detected_set | official_set),
        "consensus": sorted(detected_set & official_set),
        "detected_with_traces": sorted(detected_set | traces_set),
        "detected_with_may_contain": sorted(detected_set | may_contain_set),
        "traces_only": sorted(traces_set),
        "may_contain_only": sorted(may_contain_set)
    }

# Apply to dataframe
df["combined_allergens"] = df.apply(
    lambda row: combine_allergen_labels(
        row["detected_allergens"],
        row["official_allergens_mapped"],
        row["traces_allergens"],
        row["may_contain"]
    ), axis=1
)

# Extract consensus (only where both detected and official agree)
df["consensus_allergens"] = df["combined_allergens"].apply(lambda x: x["consensus"])

In [13]:
# Evaluate against official tags (still imperfect but useful for trend)
strict_agreement = (df["detected_allergens"].apply(set) == df["official_allergens_mapped"].apply(set)).mean()
print(f"Exact match agreement (strict): {strict_agreement:.2%}")

subset_agreement = (df["detected_allergens"].apply(set) <= df["official_allergens_mapped"].apply(set)).mean()
print(f"Detected ⊆ official (no false positives): {subset_agreement:.2%}")

superset_agreement = (df["detected_allergens"].apply(set) >= df["official_allergens_mapped"].apply(set)).mean()
print(f"Detected ⊇ official (no false negatives): {superset_agreement:.2%}")

def jaccard(set1, set2):
    if len(set1) == 0 and len(set2) == 0:
        return 1.0
    inter = len(set1 & set2)
    union = len(set1 | set2)
    return inter / union if union > 0 else 0

avg_jaccard = df.apply(lambda row: jaccard(set(row["detected_allergens"]), set(row["official_allergens_mapped"])), axis=1).mean()
print(f"Average Jaccard similarity: {avg_jaccard:.2%}")

# Per-allergen metrics
all_allergens = BIG8.keys()
stats = defaultdict(lambda: {"tp":0, "fp":0, "fn":0})

for _, row in df.iterrows():
    detected = set(row["detected_allergens"])
    official = set(row["official_allergens_mapped"])
    for a in all_allergens:
        if a in detected and a in official:
            stats[a]["tp"] += 1
        elif a in detected and a not in official:
            stats[a]["fp"] += 1
        elif a not in detected and a in official:
            stats[a]["fn"] += 1

print("\nPer-allergen metrics (after improvements):")
for a in all_allergens:
    tp = stats[a]["tp"]
    fp = stats[a]["fp"]
    fn = stats[a]["fn"]
    prec = tp / (tp+fp) if (tp+fp) > 0 else 0
    rec = tp / (tp+fn) if (tp+fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    print(f"{a:12}  Prec: {prec:.2%}  Rec: {rec:.2%}  F1: {f1:.2%}  (TP={tp}, FP={fp}, FN={fn})")

Exact match agreement (strict): 46.36%
Detected ⊆ official (no false positives): 48.82%
Detected ⊇ official (no false negatives): 97.26%
Average Jaccard similarity: 48.04%

Per-allergen metrics (after improvements):
milk          Prec: 28.76%  Rec: 92.31%  F1: 43.85%  (TP=132, FP=327, FN=11)
eggs          Prec: 4.65%  Rec: 100.00%  F1: 8.89%  (TP=4, FP=82, FN=0)
peanuts       Prec: 19.40%  Rec: 100.00%  F1: 32.50%  (TP=13, FP=54, FN=0)
tree_nuts     Prec: 21.43%  Rec: 66.67%  F1: 32.43%  (TP=6, FP=22, FN=3)
soy           Prec: 7.30%  Rec: 80.95%  F1: 13.39%  (TP=17, FP=216, FN=4)
wheat         Prec: 11.34%  Rec: 81.25%  F1: 19.90%  (TP=39, FP=305, FN=9)
fish          Prec: 14.71%  Rec: 83.33%  F1: 25.00%  (TP=10, FP=58, FN=2)
shellfish     Prec: 0.00%  Rec: 0.00%  F1: 0.00%  (TP=0, FP=37, FN=0)
coconut       Prec: 0.00%  Rec: 0.00%  F1: 0.00%  (TP=0, FP=104, FN=0)


In [14]:
# Select columns for final output
final_columns = [
    "code", "brands", "product_name_en", "ingredients_text_en",
    "detected_allergens", "official_allergens_mapped", "traces_allergens",
    "consensus_allergens", "combined_allergens", "may_contain", "coconut"
]
existing_final = [c for c in final_columns if c in df.columns]
final_df = df[existing_final].copy()
final_df.to_csv("../data/final/labeled_dataset_enhanced.csv", index=False)
print("Saved enhanced dataset with consensus labels to ../data/final/labeled_dataset_enhanced.csv")

Saved enhanced dataset with consensus labels to ../data/final/labeled_dataset_enhanced.csv
